# 03b Validate Silver Geographic Keys

Runs strict ISO-3 validation and canonical country-name reconciliation after Silver cleaning and before Gold feature assembly. Invalid codes and country/code conflicts are retained in audit reports rather than entering the Gold join.

In [1]:
%cd /content/HDX-sources-and-more-API-connection  # Move to the cloned repository.

!git fetch origin  # Download the latest branch information from GitHub.
!git checkout geo-key-hardening  # Switch from main to the Silver cleanup branch.
!git pull origin geo-key-hardening  # Make sure the branch is fully current.

!ls src  # Confirm geographic_keys.py is now present.

[Errno 2] No such file or directory: '/content/HDX-sources-and-more-API-connection # Move to the cloned repository.'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
ls: cannot access 'src': No such file or directory


In [ ]:
import sys  # Access Python's module search path.
from pathlib import Path  # Work with repository paths.

PROJECT_ROOT = Path("/content/HDX-sources-and-more-API-connection")  # Set repository root.

sys.path.insert(0, str(PROJECT_ROOT / "src"))  # Make shared modules importable.

from cleaning import clean_silver_directory  # Import the existing Silver cleaning pipeline.
from paths import CLEAN_DIR  # Import the cleaned Silver output directory.

summary_clean, rejected_clean = clean_silver_directory()  # Generate cleaned Silver CSVs.

display(summary_clean)  # Review the Silver cleaning results.

print(f"\nCleaned Silver CSVs created: {len(list(CLEAN_DIR.glob('*.csv')))}")

In [ ]:
from geographic_keys import validate_clean_directory  # Import strict geographic validation.
from paths import CLEAN_DIR, CLEAN_REPORTS_DIR  # Import canonical paths.

summary = validate_clean_directory(
    CLEAN_DIR,
    CLEAN_REPORTS_DIR
)  # Validate and canonicalize geographic keys.

display(summary)  # Review geographic validation results.

if summary.empty:  # Stop clearly if no Silver files were found.
    raise ValueError("No cleaned Silver files were available for geographic validation.")

if not summary["key_unique"].all():  # Block Gold if duplicate geographic keys remain.
    raise ValueError(
        "Geographic-key validation failed uniqueness checks; inspect reports before Gold assembly."
    )

print("Silver geographic-key validation passed. Gold assembly may proceed.")